(orchestration2)=
# Orchestration Service V2 API

This notebook demonstrates how to use the SDK to interact with the Orchestration Service V2, enabling the creation of AI-driven workflows by seamlessly integrating various modules, such as templating, large language models (LLMs), data masking and content filtering. By leveraging these modules, you can build complex, automated workflows that enhance the capabilities of your AI solutions. For more details on configuring and using these modules, please refer to the [Orchestration Service Documentation](https://help.sap.com/docs/ai-launchpad/sap-ai-launchpad/orchestration).

## Prerequisite

> **Important:** Before you begin using the SDK, make sure to set up a virtual deployment of the Orchestration Service.

For detailed guidance on setting up the Orchestration Service, please refer to the setup guide [here](https://help.sap.com/docs/ai-launchpad/sap-ai-launchpad/create-deployment-for-orchestration).

## Authentication

By default, the `OrchestrationService` initializes a `GenAIHubProxyClient`, which automatically configures credentials using configuration files or environment variables, as outlined in the *Introduction* section.

If you prefer to set credentials manually, you can provide a custom instance using the `proxy_client` parameter.

## Basic Orchestration Pipeline

Let's walk through a basic orchestration pipeline for a translation task.

### Step 1: Define the Template and Default Input Values

The `Template` class is used to define structured message templates for generating dynamic interactions with language models. In this example, the template is designed for a translation assistant, allowing users to specify a language and text for translation.

In [1]:
from gen_ai_hub.orchestration_v2 import Template, SystemMessage, UserMessage

template = Template(
    template=[
        SystemMessage(content="You are a helpful translation assistant."),
        UserMessage(content="Translate the following text to {{?to_lang}}: {{?user_query}}"),
    ],
    defaults={"to_lang": "German"}
    )

This template can be used to create translation requests where the language and text to be translated are specified dynamically. The placeholders in the `UserMessage` will be replaced with the actual values provided at runtime, and the default value for the language is set to German.

### Step 2: Define the LLM

The `LLM` class is used to configure and initialize a language model for generating text based on specific parameters. In this example, we'll use the `gpt-4o` model to perform the translation task.

**Note:** The Orchestration Service automatically manages the virtual deployment of the language model, so no additional setup is needed on your end.

In [2]:
from gen_ai_hub.orchestration_v2 import LLMModelDetails

llm = LLMModelDetails(name="gpt-5-nano", params={"max_completion_tokens": 512})

Initializes the language model to use the `gpt-5-nano` model. It will generate responses up to 512 tokens in length.

### Step 3: Create the Orchestration Configuration

The `OrchestrationConfig` class defines a configuration for integrating various modules, such as templates and language models, into a cohesive orchestration setup. It specifies how these components interact and are configured to achieve the desired operational scenario.

In [3]:
from gen_ai_hub.orchestration_v2 import PromptTemplatingModuleConfig, ModuleConfig, OrchestrationConfig

prompt_template = PromptTemplatingModuleConfig(prompt=template,
                                               model=llm)

module_config = ModuleConfig(prompt_templating=prompt_template)

config = OrchestrationConfig(modules=module_config)

### Step 4: Run the Orchestration Request

The `OrchestrationService` class is used to interact with a orchestration service instance by providing configuration details to initiate and manage its operations.

In [4]:
from gen_ai_hub.orchestration_v2 import OrchestrationService

orchestration_service = OrchestrationService(config=config)

Call the `run` method with the required `placeholder values`. The service will process the input according to the configuration and return the result.

In [ ]:
result = orchestration_service.run(placeholder_values={"user_query": "The Orchestration Service is working!"})
print(result.final_result.choices[0].message.content)

(prompt_registry)=
#### Referencing Templates in the Prompt Registry
 In Step 3 you can also use a prompt template reference, which allows you to reuse existing templates stored in the Prompt Registry.

In [6]:
from gen_ai_hub.orchestration_v2 import TemplateRefByID, TemplateRefByScenarioNameVersion

template_by_id = TemplateRefByID(id="648871d9-b207-441c-8c13-afee71b0dbec") # this is just an example id
template_by_names = TemplateRefByScenarioNameVersion(scenario="translation", name="translate_text", version="0.1.0")

(response_format)=
#### Overview of response_format Parameter Options

The `response_format` parameter allows the model output to be formatted in several predefined ways, as follows:

1. **text**: This is the simplest form where the model's output is generated as plain text. It is suitable for applications that require raw text processing.

2. **json_object**: Under this setting, the model's output is structured as a JSON object. This is useful for applications that handle data in JSON format, enabling easy integration with web applications and APIs.

3. **json_schema**: This setting allows the model's output to adhere to a defined JSON schema. This is particularly useful for applications that require strict data validation, ensuring the output matches a predefined schema.

In [7]:
from gen_ai_hub.orchestration_v2 import SystemMessage, UserMessage, Template, ResponseFormatText

template = Template(
    template=[
        SystemMessage(content="You are a helpful translation assistant."),
        UserMessage(content="{{?user_query}}")
    ],
    response_format=ResponseFormatText(),
    defaults={"user_query": "Who was the first person on the moon?"}
)

# Response:
# The first man on the moon was Neil Armstrong.

In [8]:
from gen_ai_hub.orchestration_v2 import SystemMessage, UserMessage, Template, ResponseFormatJsonObject

template = Template(
    template=[
        SystemMessage(content="You are a helpful translation assistant. Format the response as json."),
        UserMessage(content="{{?user_query}}")
    ],
    response_format=ResponseFormatJsonObject(),
    defaults={"user_query": "Who was the first person on the moon?"}
)

# Response:
# {
#     "First_man_on_the_moon": "Neil Armstrong"
# }

**Important:** When using `response_format` as json_object, ensure that messages contain the word 'json' in some form.

In [9]:
from gen_ai_hub.orchestration_v2 import SystemMessage, UserMessage, Template, ResponseFormatJsonSchema, JSONResponseSchema

json_schema = {
    "title": "Person",
    "type": "object",
    "properties": {
            "firstName": {
            "type": "string",
            "description": "The person's first name."
        },
            "lastName": {
            "type": "string",
            "description": "The person's last name."
        }
    }
}
template = Template(
    template=[
        SystemMessage(content="You are a helpful translation assistant. Format the response as json."),
        UserMessage(content="{{?user_query}}")
    ],
    response_format=ResponseFormatJsonSchema(
                json_schema=JSONResponseSchema(
                    name="person", description="person mapping", schema=json_schema
                ),
            ),
    defaults={"user_query": "Who was the first person on the moon?"}
)

# Response:
# {
#     "firstName": "Neil",
#     "lastName": "Armstrong"
# }

(orchestration_deployment)=
## Understanding Deployment Resolution

The `OrchestrationService` class provides multiple ways to specify and target orchestration deployments when sending requests. Below are the available options:

### Default Behavior

If no parameters are provided, the `OrchestrationService` automatically searches for a `RUNNING` deployment. If multiple running deployments exist, the service selects the most recently created one.

### Direct Deployment Specification

You can explicitly define the target deployment using the following options:

1. **API URL** (`api_url`):
    - Specify the exact URL assigned to the deployment during its creation.
    - Refer to the Prerequisites section for more details on obtaining the deployment URL.

2. **Deployment ID** (`deployment_id`):
    - Use the unique identifier assigned to the deployment instead of the URL.

### Config-Based Specification

If you want to target deployments based on their configuration source, use one of the following options:

1. **Configuration ID** (`config_id`):
    - The `OrchestrationService` searches for a `RUNNING` deployment created using the provided configuration ID.

2. **Configuration Name** (`config_name`):
    - The service looks for a `RUNNING` deployment that matches the specified configuration name.

If multiple deployments match the given configuration criteria, the most recently created one will be selected automatically.

## Optional Modules

### Data Masking

The `Data Masking` module `anonymizes` or `pseudonymizes` personally identifiable information (PII) before it is processed by the LLM module. Currently, `SAPDataPrivacyIntegration` is the only available masking provider.

#### Masking Types

- **Anonymization**: All identifying information is replaced with placeholders (e.g., MASKED_ENTITY), and the original data cannot be recovered, ensuring that no trace of the original information is retained.
- **Pseudonymization**: Data is substituted with unique placeholders (e.g., MASKED_ENTITY_ID), allowing the original information to be restored if needed.

In both cases, the masking module identifies sensitive data and replaces it with appropriate placeholders before further processing.

(allow_list)=

#### Configuration Options

- **entities**: Specify which types of entities to mask (e.g., EMAIL, PHONE, PERSON).
- **allowlist**: Provide specific terms or patterns that should be excluded from masking, even if they match entity types.
- **mask_grounding_input**: When enabled, ensures that masking is also applied to the context provided to the grounding module.

In [10]:
from gen_ai_hub.orchestration_v2.utils import load_text_file
from gen_ai_hub.orchestration_v2 import (SystemMessage, UserMessage, Template, PromptTemplatingModuleConfig,
                                         LLMModelDetails, ModuleConfig, OrchestrationConfig, OrchestrationService,
                                         MaskingModuleConfig, MaskingProviderConfig, MaskingMethod, DPIStandardEntity,
                                         ProfileEntity)

orchestration_service = OrchestrationService()

data_masking_config = MaskingModuleConfig(
    providers=[MaskingProviderConfig(
        method=MaskingMethod.ANONYMIZATION,
        entities=[
            DPIStandardEntity(type=ProfileEntity.ADDRESS),
            DPIStandardEntity(type=ProfileEntity.EMAIL),
            DPIStandardEntity(type=ProfileEntity.PHONE),
            DPIStandardEntity(type=ProfileEntity.PERSON),
        ],
        allowlist=["M&K Group"],  # Terms to exclude from masking
    )],

)

template = Template(
    template=[
        SystemMessage(content="You are a helpful AI assistant."),
        UserMessage(content="Summarize the following CV in 10 sentences: {{?orgCV}}"),
    ]
    )

llm=LLMModelDetails(name="gpt-4o")

prompt_template = PromptTemplatingModuleConfig(prompt=template,
                                               model=llm)

module_config = ModuleConfig(prompt_templating=prompt_template, masking=data_masking_config)

config = OrchestrationConfig(modules=module_config)

cv_as_string = load_text_file("data/cv.txt")

result = orchestration_service.run(
    config=config,
    placeholder_values={"orgCV": cv_as_string}
)

In [ ]:
print(result.final_result.choices[0].message.content)

(content_filtering)=
### Content Filtering

The `Content Filtering` module can be configured to filter both the `input` to the LLM module (input filter) and the `output` generated by the LLM (output filter). The module uses predefined classification services to detect inappropriate or unwanted content. Azure Content Filter sensitivity is controlled by customizable `thresholds`, assuring the content aligns with the desired standards before processing or generating as output. Llama Guard 3 Filter, equipped with 14 categories, runs on a binary mechanism, accepting only true or false. Setting a category to true enables filtering for it. It's possible to execute both filters in a single request, optimizing efficiency.

In [11]:
from gen_ai_hub.orchestration_v2 import (AzureContentSafetyInput, AzureContentSafetyOutput, AzureThreshold,
                                         LlamaGuard38bFilter, FilteringModuleConfig, InputFiltering, OutputFiltering,
                                         AzureContentSafetyInputFilterConfig, AzureContentSafetyOutputFilterConfig,
                                         LlamaGuard38bFilterConfig)

content_filter_config = FilteringModuleConfig(
    input=InputFiltering(filters=[
        AzureContentSafetyInputFilterConfig(config=AzureContentSafetyInput(hate=AzureThreshold.ALLOW_SAFE,
                                                                                  violence=AzureThreshold.ALLOW_SAFE,
                                                                                  self_harm=AzureThreshold.ALLOW_SAFE,
                                                                                  sexual=AzureThreshold.ALLOW_SAFE)),
        LlamaGuard38bFilterConfig(config=LlamaGuard38bFilter(hate=True))
        ]),
    output=OutputFiltering(filters=[
        AzureContentSafetyOutputFilterConfig(config=AzureContentSafetyOutput(hate=AzureThreshold.ALLOW_SAFE,
                                                                                  violence=AzureThreshold.ALLOW_SAFE,
                                                                                  self_harm=AzureThreshold.ALLOW_SAFE,
                                                                                  sexual=AzureThreshold.ALLOW_SAFE)),
        LlamaGuard38bFilterConfig(config=LlamaGuard38bFilter(hate=True))
    ])

)

template = Template(
    template=[
        SystemMessage(content="You are a helpful AI assistant."),
        UserMessage(content="{{?text}}"),
    ]
    )

llm=LLMModelDetails(name="gpt-4o")

prompt_template = PromptTemplatingModuleConfig(prompt=template,
                                               model=llm)

module_config = ModuleConfig(prompt_templating=prompt_template, filtering=content_filter_config)

config = OrchestrationConfig(modules=module_config)

client = OrchestrationService(config=config)

In [ ]:
from gen_ai_hub.orchestration_v2 import OrchestrationError

try:
    result = client.run(placeholder_values={"text": "I hate you"})
    print(result.final_result.choices[0].message.content)
except OrchestrationError as er:
    print(er.message)

(orchestration_streaming)=
## Streaming

When you initiate an orchestration request, the full response is typically processed and delivered in one go. For longer responses, this can lead to delays in receiving the complete output. To mitigate this, you have the option to stream the results as they are being generated. This helps in rapidly processing or displaying initial portions of the results without waiting for the entire computation to finish.


To activate streaming, use the `stream` method of the `OrchestrationService` with the `stream` option in `OrchestrationConfig`. This method returns an object that streams chunks of the response as they become available. You can then extract relevant information from the `delta` field.

Here's how you can set up a simple configuration to stream orchestration results:

In [ ]:
from gen_ai_hub.orchestration_v2 import GlobalStreamOptions

template = Template(
    template=[
        SystemMessage(content="You are a helpful AI assistant."),
        UserMessage(content="{{?text}}"),
    ]
    )

llm=LLMModelDetails(
        name="gpt-4o-mini",
        params={
            "max_completion_tokens": 256,
            "temperature": 0.0
        }
    )

prompt_template = PromptTemplatingModuleConfig(prompt=template,
                                               model=llm)

module_config = ModuleConfig(prompt_templating=prompt_template)

config = OrchestrationConfig(modules=module_config,
                             stream=GlobalStreamOptions(enabled=True))

client = OrchestrationService(config=config)

result = client.stream(placeholder_values={
    "text": "Which color is the sky? Answer in one sentence."
})
for part in result:
    print(part.final_result.choices[0].delta.content)
    print("*" * 20)


**Note:** As shown above, streaming responses contain a delta field instead of a message field.

You can customize the global stream behavior by setting options like `chunk_size` which controls the amount of data processed in each chunk:

In [ ]:
config = OrchestrationConfig(modules=module_config,
                             stream=GlobalStreamOptions(enabled=True, chunk_size=25))

client = OrchestrationService(config=config)

result = client.stream(placeholder_values={
    "text": "Which color is the sky? Answer in one sentence."
})
for part in result:
    print(part.final_result.choices[0].delta.content)
    print("*" * 20)

Modules that influence or process streaming results, such as `OutputFiltering`, might need specific stream options. The `overlap` option allows you to include extra context during the filtering process:

In [ ]:
from gen_ai_hub.orchestration_v2 import FilteringStreamOptions
content_filter_config = FilteringModuleConfig(
    output=OutputFiltering(
        filters=[AzureContentSafetyOutputFilterConfig(config=AzureContentSafetyOutput(hate=0))],
        stream_options=FilteringStreamOptions(overlap=20))
)

template = Template(
    template=[
        SystemMessage(content="You are a helpful AI assistant."),
        UserMessage(content="{{?text}}"),
    ]
    )

llm=LLMModelDetails(name="gpt-4o")

prompt_template = PromptTemplatingModuleConfig(prompt=template,
                                               model=llm)

module_config = ModuleConfig(prompt_templating=prompt_template, filtering=content_filter_config)

config = OrchestrationConfig(modules=module_config,
                             stream=GlobalStreamOptions(enabled=True))

client = OrchestrationService(config=config)

response = client.stream(placeholder_values={"text": "Which color is the sky? Answer in one sentence."})

for chunk in response:
    print(chunk.final_result.choices[0].delta.content, end='')


(tool_calling)=
## Tool Calling (Function Calling)

The Orchestration Service supports **tool calling**, which allows large language models (LLMs) to request the execution of external operations such as Python functions, API calls, or other tools as part of their workflow.

This feature enables you to build advanced AI solutions that can perform calculations, access data, or interact with external systems in response to user input.

---

### Defining Tools

You can define tools in several ways, depending on your requirements and the level of control you need.

#### Using the Python Decorator

The simplest way to define a tool is to decorate a Python function with `@function_tool()`. The function’s signature and docstring are used to describe the tool to the LLM.

In [14]:
from gen_ai_hub.orchestration_v2 import function_tool

@function_tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

@function_tool()
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

tools = [multiply, add]

#### Using the `FunctionTool` Class

For more control, you can use the `FunctionTool` class directly. This is useful if you want to customize the schema, enable strict argument checking, or wrap an existing function.

In [15]:
from gen_ai_hub.orchestration_v2 import FunctionTool, FunctionObject

def get_weather(location: str) -> str:
    """Get current temperature for a given location."""
    # Replace with your actual implementation
    return "22°C"

weather_tool_func = FunctionObject(
    name="get_weather",
    description="Get current temperature for a given location.",
    parameters={
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "City and country e.g. Bogotá, Colombia"
            }
        },
        "required": ["location"],
        "additionalProperties": False
    },
    strict=True,
    function=get_weather
)

weather_tool = FunctionTool(function=weather_tool_func)

tools = [weather_tool]

You can also create a `FunctionTool` from a function using the `from_function` static method:

In [ ]:
weather_tool = FunctionTool.from_function(get_weather, strict=True)
tools = [weather_tool]

#### Using a JSON Schema Dictionary

You can define a tool directly as a JSON schema dictionary. This is useful if you want to specify the tool interface without implementing the function in Python, or if you want to integrate with external systems.

In [ ]:
tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current temperature for a given location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City and country e.g. Bogotá, Colombia"
                }
            },
            "required": [
                "location"
            ],
            "additionalProperties": False
        },
        "strict": True
    }
}]

You can then attach any of these tool definitions to your template:

In [16]:
from gen_ai_hub.orchestration_v2 import Template, SystemMessage, UserMessage

template = Template(
    template=[
        SystemMessage(content="You are a weather assistant."),
        UserMessage(content="What is the temperature in {{?location}}?"),
    ],
    tools=tools,
)

### Synchronous Tool Call Workflow

When the LLM decides to call a tool, the orchestration response will include a `tool_calls` field. You are responsible for executing the tool(s), adding the results to the conversation history, and running the orchestration again to get the final answer.

In [ ]:
from typing import List
from gen_ai_hub.orchestration_v2 import ChatMessage, SystemMessage, UserMessage, ToolChatMessage


# Assume 'template' and 'weather_tool' are defined as above
llm = LLMModelDetails(name="gpt-4o-mini", params={"max_completion_tokens": 200, "temperature": 0.0})
rompt_template = PromptTemplatingModuleConfig(prompt=template,
                                               model=llm)
module_config = ModuleConfig(prompt_templating=prompt_template)

config = OrchestrationConfig(modules=module_config)

client = OrchestrationService(config=config)
template_values = {"location": "Bogotá, Colombia"}

# First run: triggers tool call
service = OrchestrationService()
response = service.run(config=config, placeholder_values=template_values)
tool_calls = response.final_result.choices[0].message.tool_calls

# Execute tool(s) and build new history
history: List[ChatMessage] = []
history.extend(response.intermediate_results.templating)
history.append(response.final_result.choices[0].message)

for tool_call in tool_calls:
    # For FunctionTool, use .execute(**tool_call.function.parse_arguments())
    result = weather_tool.execute(**tool_call.function.parse_arguments())
    tool_message = ToolChatMessage(
        content=str(result),
        tool_call_id=tool_call.id,
    )
    history.append(tool_message)

# Second run: LLM receives tool result and produces final answer
response2 = service.run(
    config=config,
    placeholder_values=template_values,
    history=history,
)
print(response2.final_result.choices[0].message.content)

### Streaming Tool Calls

When using streaming, tool calls may be split across multiple chunks. The `delta.tool_calls` field in each chunk contains partial or complete tool call information. You may need to buffer and concatenate arguments if they arrive in pieces.

In [ ]:
# Assume 'config' and 'service' are defined as above
config = OrchestrationConfig(modules=module_config, stream=GlobalStreamOptions(enabled=True))
service = OrchestrationService()
stream = service.stream(config=config, placeholder_values=template_values)

final_tool_calls = {}

for chunk in stream:
    for tool_call in chunk.final_result.choices[0].delta.tool_calls or []:
        index = tool_call.index
        if index not in final_tool_calls:
            final_tool_calls[index] = tool_call
        else:
            # Concatenate arguments if split across chunks
            final_tool_calls[index].function.arguments += tool_call.function.arguments

# Now final_tool_calls contains all tool calls with complete arguments

**⚠️ Note on Agentic Loop Support:**

> The current SDK **does not provide built-in abstractions or convenience methods for managing the agentic loop** (the process of automatically handling tool call detection, execution, and iterative orchestration until a final answer is produced).
>
> As a user, you are responsible for:
> - Detecting tool calls in the LLM response
> - Executing the corresponding Python functions
> - Appending tool results to the conversation history (as `ToolMessage`)
> - Re-invoking the orchestration service as needed
>
> This approach gives you maximum flexibility, but you must implement the orchestration loop logic yourself.

(input_images)=
## Using Images as Input

The Orchestration Service supports multimodal prompts, enabling you to include images alongside text in your messages. This powerful feature unlocks a variety of applications, such as visual question answering (VQA), image captioning, object recognition, and generating text creatively based on visual input.

This guide details how to prepare image inputs, integrate them into your prompts, and execute the orchestration to get insightful responses.

### 1. Preparing Image Inputs

To use an image, you first need to represent it as an `ImageItem` object. The `gen_ai_hub.orchestration.models.multimodal_items.ImageItem` class provides two convenient ways to do this:

#### a) From a URL or Data URL

This method is ideal for images hosted online or when you have the image data encoded as a Data URL (base64 encoded).

*   **Standard URL:** Provide a direct web link to the image file.
*   **Data URL:** Provide the image data directly embedded in the URL string.

**Note:** For web URLs, ensure the image is publicly accessible, as the service will need to fetch it.

In [18]:
from gen_ai_hub.orchestration_v2 import ImageItem

# Example 1: Image from a standard, publicly accessible URL
# Ensure the URL points directly to the image file (e.g., .png, .jpg, ...)
image_from_web = ImageItem(url="https://picsum.photos/id/1/200/300")  # example image URL

# Example 2: Image from a Data URL (base64-encoded)
# This is useful when you have the image content as a string.
# The format is "data:[<mediatype>][;base64],<data>"
image_from_data_url = ImageItem(
    url="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAoAAAAKCAIAAAACUFjqAAAAE0lEQVR4nGP8z4APMOGVZRip0gBBLAETee26JgAAAABJRU5ErkJggg=="
)

#### b) From a Local File

If your image resides on your local filesystem, you can load it directly using the `ImageItem.from_file()` class method.
The `from_file` method handles opening, reading, and base64 encoding the image data for you, packaging it into an `ImageItem`.

In [ ]:
from gen_ai_hub.orchestration_v2 import ImageItem

# Example: Image from a local file path
# To use this 'image_from_local_file' object, ensure it was successfully created.
try:
    image_from_local_file = ImageItem.from_file("path/to/your/local/image.jpeg")
except FileNotFoundError:
    print("Error: The specified image file was not found.")
except Exception as e:
    print(f"An error occurred while loading the image: {e}")

### 2. Adding Images to a Prompt

Once you have your `ImageItem` object(s), you can combine them with text to create a multimodal prompt. This is done by passing a list containing `ImageItem` instances and text strings to the `content` parameter of a `UserMessage`.

In [ ]:
from gen_ai_hub.orchestration_v2 import UserMessage

llm=LLMModelDetails(name="gpt-4o")

# Simple visual question answering
content_vqa = [image_from_web, "What objects are prominent in this image?"]

# Create a UserMessage with the mixed content
user_message = UserMessage(content=content_vqa)

# Create a Template containing the UserMessage
prompt_template = PromptTemplatingModuleConfig(prompt=template,model=llm)

module_config = ModuleConfig(prompt_templating=prompt_template)

config = OrchestrationConfig(modules=module_config)
service = OrchestrationService(config=config)
response = service.run()
print(response.final_result.choices[0].message.content)

(translation)=
## Translation
Translation module can be  used to translate text from one language to another. You can use this module to translate input text before it is processed by the LLM module, or to translate the output generated by the LLM module. The translation module uses the SAP Document Translation service to perform the translation.

In [20]:
from gen_ai_hub.orchestration_v2 import (SystemMessage, UserMessage, Template, PromptTemplatingModuleConfig,
                                         LLMModelDetails, ModuleConfig, OrchestrationConfig, OrchestrationService,
                                         TranslationModuleConfig, SAPDocumentTranslationInput,
                                         SAPDocumentTranslationOutput, InputTranslationConfig, OutputTranslationConfig)

translation_config = TranslationModuleConfig(
    input=SAPDocumentTranslationInput(
        config=InputTranslationConfig(
            source_language="en-US",
            target_language="de-DE"
        )
    ),
    output=SAPDocumentTranslationOutput(
        config=OutputTranslationConfig(
            source_language="de-DE",
            target_language="fr-FR"
        )
    )
)


template = Template(
    template=[
        SystemMessage(content="You are a helpful AI assistant."),
        UserMessage(content="{{?text}}"),
    ]
    )

llm=LLMModelDetails(name="gpt-4o")

prompt_template = PromptTemplatingModuleConfig(prompt=template,
                                               model=llm)

module_config = ModuleConfig(prompt_templating=prompt_template, translation=translation_config)

config = OrchestrationConfig(modules=module_config)

orchestration_service = OrchestrationService()

In [ ]:
result = orchestration_service.run(
    config=config,
    placeholder_values={"text": "What is the capital of Germany?"}
)

In [ ]:
print(result.final_result.choices[0].message.content)

## Advanced Examples

In [ ]:
service = OrchestrationService(api_url=YOUR_API_URL)

### Translation Service

This example extends the initial walkthrough of a basic orchestration pipeline by abstracting the translation task into its own reusable `TranslationService` class. Once the configuration is established, it can be easily adapted and reused for different translation scenarios.

In [21]:
from gen_ai_hub.orchestration_v2 import (OrchestrationConfig, ModuleConfig, LLMModelDetails, SystemMessage, UserMessage,
                                         Template, PromptTemplatingModuleConfig, OrchestrationService)


class TranslationService:
    def __init__(self, orchestration_service: OrchestrationService):
        self.service = orchestration_service
        self.template = Template(
                template=[
                    SystemMessage(content="You are a helpful AI assistant."),
                    UserMessage(content="Translate the following text to {{?to_lang}}: {{?text}}"),
                ],
                defaults={"to_lang": "en-US"}
                )

        self.llm=LLMModelDetails(name="gpt-4o")

        self.prompt_template = PromptTemplatingModuleConfig(prompt=self.template, model=self.llm)

        self.module_config = ModuleConfig(prompt_templating=self.prompt_template)

        self.config = OrchestrationConfig(modules=self.module_config)

    def translate(self, text, to_lang):
        response = self.service.run(
            config=self.config,
            placeholder_values={
                "to_lang": to_lang,
                "text": text
            },
        )

        return response.final_result.choices[0].message.content


In [ ]:
translator = TranslationService(orchestration_service=service)

In [ ]:
result = translator.translate(text="Hello, world!", to_lang="French")
print(result)

In [ ]:
result = translator.translate(text="Hello, world!", to_lang="Spanish")
print(result)

In [ ]:
result = translator.translate(text="Hello, world!", to_lang="German")
print(result)

### Chatbot with Memory

This example demonstrates how to integrate the `OrchestrationService` with a chatbot to handle conversational flow.

When making requests to the orchestration service, you can specify a list of messages as `history` that will be prepended to the templated content and processed by the templating module. These messages are plain, non-templated messages, as they typically represent past conversation outputs — such as in this chatbot scenario.

It’s important to note that managing conversation history / state is handled locally in the `ChatBot` class, not by the orchestration service itself.

In [22]:
from typing import List

from gen_ai_hub.orchestration_v2 import (OrchestrationConfig, ModuleConfig, LLMModelDetails, ChatMessage, SystemMessage,
                                         UserMessage, Template, PromptTemplatingModuleConfig, OrchestrationService)


class ChatBot:
    def __init__(self, orchestration_service: OrchestrationService):
        self.service = orchestration_service
        self.template = Template(
            template=[
                SystemMessage(content="You are a helpful chatbot assistant."),
                UserMessage(content="{{?user_query}}")
            ]
        )

        self.llm = LLMModelDetails(name="gpt-4o")

        self.prompt_template = PromptTemplatingModuleConfig(prompt=self.template, model=self.llm)

        self.module_config = ModuleConfig(prompt_templating=self.prompt_template)

        self.config = OrchestrationConfig(modules=self.module_config)

        self.history: List[ChatMessage] = []

    def chat(self, user_input):
        response = self.service.run(
            config=self.config,
            placeholder_values={"user_query": user_input},
            history=self.history,
        )

        message = response.final_result.choices[0].message

        self.history = response.intermediate_results.templating
        self.history.append(message)

        return message.content

    def reset(self):
        self.history = []

In [ ]:
bot = ChatBot(orchestration_service=OrchestrationService())

In [ ]:
print(bot.chat("Hello, how are you?"))

In [ ]:
print(bot.chat("What's the weather like today?"))

In [ ]:
print(bot.chat("Can you remember what I first asked you?"))

In [ ]:
bot.reset()

In [ ]:
print(bot.chat("Can you remember what I first asked you?"))

### Sentiment Analysis with Few Shot Learning 

This example demonstrates the different message `roles` in the templating module through a few-shot learning use case with the `FewShotLearner` class.

- **Message Types:** Different message types (`SystemMessage`, `UserMessage`, `AssistantMessage`) structure the interaction and guide the model's behavior.
- **Templating:** The template includes these examples, ending with a `placeholder` ({{?user_input}}) for dynamic user input.
- **Few-Shot Examples:** Pairs of UserMessage and AssistantMessage show how the model should respond to similar queries.


The FewShotLearner class manages the dynamic creation of the template and ensures the correct message roles are used for each user input.

In [23]:
from typing import List, Tuple

from gen_ai_hub.orchestration_v2 import (OrchestrationConfig,ModuleConfig, LLMModelDetails, SystemMessage, UserMessage,
                                         AssistantMessage, Template, PromptTemplatingModuleConfig, OrchestrationService)


class FewShotLearner:
    def __init__(
            self,
            orchestration_service: OrchestrationService,
            system_message: SystemMessage,
            examples: List[Tuple[UserMessage, AssistantMessage]],
    ):
        self.service = orchestration_service


        self.llm = LLMModelDetails(name="gpt-4o-mini")

        self.prompt_template = PromptTemplatingModuleConfig(
            prompt=self._create_few_shot_template(system_message, examples),
            model=self.llm
        )

        self.module_config = ModuleConfig(prompt_templating=self.prompt_template)

        self.config = OrchestrationConfig(modules=self.module_config)

    @staticmethod
    def _create_few_shot_template(
            system_message: SystemMessage,
            examples: List[Tuple[UserMessage, AssistantMessage]],
    ) -> Template:
        messages = [system_message]

        for example in examples:
            messages.append(example[0])
            messages.append(example[1])
        messages.append(UserMessage(content="{{?user_input}}"))

        return Template(template=messages)

    def predict(self, user_input: str) -> str:
        response = self.service.run(
            config=self.config,
            placeholder_values={"user_input": user_input},
        )

        return response.final_result.choices[0].message.content

In [ ]:
sentiment_examples = [
    (UserMessage(content="I love this product!"), AssistantMessage(content="Positive")),
    (UserMessage(content="This is terrible service."), AssistantMessage(content="Negative")),
    (UserMessage(content="The weather is okay today."), AssistantMessage(content="Neutral")),
]

In [ ]:
sentiment_analyzer = FewShotLearner(
    orchestration_service=OrchestrationService(),
    system_message=SystemMessage(
        content="You are a sentiment analysis assistant. Classify the sentiment as Positive, Negative, or Neutral."
    ),
    examples=sentiment_examples,
)

In [ ]:
print(sentiment_analyzer.predict("The movie was a complete waste of time!"))

In [ ]:
print(
    sentiment_analyzer.predict("The traffic was fortunately unusually light today.")
)

In [ ]:
print(
    sentiment_analyzer.predict("I'm not sure how I feel about the recent events.")
)

(orchestration_async)=
## Async Support

The `OrchestrationService` also supports asynchronous calls.
Use:
- `arun` from the async version of `run`
- `astream` from the async version of `stream`

In [24]:
from gen_ai_hub.orchestration_v2 import (SystemMessage, UserMessage, Template, PromptTemplatingModuleConfig,
                                         LLMModelDetails, OrchestrationConfig, ModuleConfig)

from IPython.display import display, Markdown # just for pretty print in jupyter

template = Template(
    template=[
        SystemMessage(content="This is a system message."),
        UserMessage(content="Write a markdown cheatsheet!"),
    ]
    )

llm=LLMModelDetails(name="gemini-2.0-flash")

prompt_template = PromptTemplatingModuleConfig(prompt=template,
                                               model=llm)

module_config = ModuleConfig(prompt_templating=prompt_template)

config = OrchestrationConfig(modules=module_config)

# Instantiate the orchestration service.
from gen_ai_hub.orchestration_v2 import OrchestrationService
orchestration_service = OrchestrationService(config=config)


In [ ]:
async def test_async():
    async_result = await orchestration_service.arun()
    display(Markdown(async_result.final_result.choices[0].message.content))

await test_async()

In [ ]:
from gen_ai_hub.orchestration_v2 import GlobalStreamOptions

config_stream = OrchestrationConfig(modules=module_config,stream=GlobalStreamOptions(enabled=True))

async def test_streaming_async():
    streamed_content = ""
    async for chunk in await orchestration_service.astream(config=config_stream):
        streamed_content += chunk.final_result.choices[0].delta.content
    display(Markdown(streamed_content))

await test_streaming_async()

(orchestration_embeddings)=
## Embeddings

The Orchestration Service provides an embeddings endpoint for generating vector representations of text. Embeddings capture the semantic meaning of text, enabling powerful applications like semantic search, document clustering, and retrieval-augmented generation (RAG).

**Key Use Cases:**
- **Semantic Search**: Find documents based on meaning, not just keywords
- **RAG (Retrieval-Augmented Generation)**: Retrieve relevant context for LLM prompts
- **Document Clustering**: Group similar documents together
- **Similarity Comparison**: Measure how semantically similar two texts are

### Basic Usage

Generate an embedding for a single text string with minimal configuration.

In [ ]:
from gen_ai_hub.orchestration_v2 import (OrchestrationService, EmbeddingsOrchestrationConfig, EmbeddingsModuleConfigs,
                                         EmbeddingsModelConfig, EmbeddingsModelDetails, EmbeddingsInput)

service = OrchestrationService()

# Minimal configuration - just specify the model
embeddings_config = EmbeddingsOrchestrationConfig(
    modules=EmbeddingsModuleConfigs(
        embeddings=EmbeddingsModelConfig(
            model=EmbeddingsModelDetails(name="text-embedding-3-large")
        )
    )
)

response = service.embed(
    config=embeddings_config,
    input=EmbeddingsInput(text="Hello World!")
)

embedding = response.final_result.data[0].embedding
print(f"Embedding dimensions: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

### Customizing Embedding Parameters

You can customize the embedding output with parameters like `dimensions`, `encoding_format`, and `normalize`.

| Parameter | Description | Values                      |
|-----------|-------------|-----------------------------|
| `dimensions` | Number of dimensions in the output | e.g. 256, 512, 1536, 3072   |
| `encoding_format` | Output format | `FLOAT`, `BASE64`, `BINARY` |
| `normalize` | Normalize the vector | `True`, `False`             |

In [ ]:
from gen_ai_hub.orchestration_v2 import EmbeddingsModelParams, EmbeddingsEncodingFormat

embeddings_config_custom = EmbeddingsOrchestrationConfig(
    modules=EmbeddingsModuleConfigs(
        embeddings=EmbeddingsModelConfig(
            model=EmbeddingsModelDetails(
                name="text-embedding-3-large",
                params=EmbeddingsModelParams(
                    dimensions=256,  # Reduce dimensions for efficiency
                    encoding_format=EmbeddingsEncodingFormat.FLOAT,
                    normalize=True
                )
            )
        )
    )
)

response = service.embed(
    config=embeddings_config_custom,
    input=EmbeddingsInput(text="Hello World!")
)

print(f"Embedding dimensions: {len(response.final_result.data[0].embedding)}")

### Batch Embeddings

Generate embeddings for multiple texts in a single request for better efficiency.

In [ ]:
documents = [
    "Artificial intelligence is transforming industries worldwide.",
    "Machine learning models require large amounts of training data.",
    "Neural networks are inspired by the human brain structure.",
    "Deep learning has achieved breakthroughs in image recognition."
]

response = service.embed(
    config=embeddings_config,
    input=EmbeddingsInput(text=documents)
)

print(f"Generated {len(response.final_result.data)} embeddings")
for result in response.final_result.data:
    print(f"  Index {result.index}: {len(result.embedding)} dimensions")

### Input Type Hints (Asymmetric Search)

Some embedding models support asymmetric search, where queries and documents are embedded differently for better retrieval. Use the `type` parameter to hint the purpose of your text.

| Type | Use Case |
|------|----------|
| `TEXT` | General purpose (default) |
| `DOCUMENT` | Content to be indexed and searched |
| `QUERY` | Search queries to find relevant documents |

In [29]:
from gen_ai_hub.orchestration_v2 import EmbeddingsInputType

# Embed a document for storage in a vector database
doc_response = service.embed(
    config=embeddings_config,
    input=EmbeddingsInput(
        text="SAP is a German multinational software company that develops enterprise software.",
        type=EmbeddingsInputType.DOCUMENT
    )
)

# Embed a query for searching
query_response = service.embed(
    config=embeddings_config,
    input=EmbeddingsInput(
        text="What is SAP?",
        type=EmbeddingsInputType.QUERY
    )
)

### Embeddings with Data Masking

When embedding sensitive data, use the data masking module to anonymize PII before generating embeddings. This ensures sensitive information is not exposed to the embedding model.

In [ ]:
from gen_ai_hub.orchestration_v2 import (MaskingModuleConfig, MaskingMethod, MaskingProviderConfig, DPIStandardEntity,
                                         ProfileEntity)

embeddings_config_with_masking = EmbeddingsOrchestrationConfig(
    modules=EmbeddingsModuleConfigs(
        embeddings=EmbeddingsModelConfig(
            model=EmbeddingsModelDetails(name="text-embedding-3-large")
        ),
        masking=MaskingModuleConfig(
            masking_providers=[
                MaskingProviderConfig(
                    method=MaskingMethod.ANONYMIZATION,
                    entities=[
                        DPIStandardEntity(type=ProfileEntity.PERSON),
                        DPIStandardEntity(type=ProfileEntity.EMAIL),
                        DPIStandardEntity(type=ProfileEntity.PHONE),
                    ]
                )
            ]
        )
    )
)

response = service.embed(
    config=embeddings_config_with_masking,
    input=EmbeddingsInput(
        text="Contact John Smith at john.smith@example.com or call 555-123-4567."
    )
)

print(f"Embedding generated with PII masked")
print(f"Intermediate results: {response.intermediate_results}")
print(f"Dimensions: {len(response.final_result.data[0].embedding)}")

#### Masking with Custom Entities and Allowlist

Use regular expressions to mask custom patterns and allowlists to exclude specific terms from masking.

In [31]:
from gen_ai_hub.orchestration_v2 import DPICustomEntity, DPIMethodConstant

embeddings_config_advanced_masking = EmbeddingsOrchestrationConfig(
    modules=EmbeddingsModuleConfigs(
        embeddings=EmbeddingsModelConfig(
            model=EmbeddingsModelDetails(name="text-embedding-3-large")
        ),
        masking=MaskingModuleConfig(
            masking_providers=[
                MaskingProviderConfig(
                    method=MaskingMethod.ANONYMIZATION,
                    entities=[
                        DPIStandardEntity(type=ProfileEntity.PERSON),
                        DPIStandardEntity(type=ProfileEntity.ORG),
                        # Custom pattern for internal IDs like "89-SAP-550"
                        DPICustomEntity(
                            regex=r"\b[0-9]{2}-SAP-[0-9]{3}\b",
                            replacement_strategy=DPIMethodConstant(
                                method="constant",
                                value="REDACTED_ID"
                            )
                        ),
                    ],
                    # These terms will NOT be masked
                    allowlist=["SAP", "Microsoft"]
                )
            ]
        )
    )
)

response = service.embed(
    config=embeddings_config_advanced_masking,
    input=EmbeddingsInput(
        text="Employee John Doe (ID: 89-SAP-550) works at SAP with Microsoft partners."
    )
)

### Async Embeddings

For non-blocking operations, use the async `aembed` method.

In [ ]:
async def embed_async():
    async_service = OrchestrationService()

    response = await async_service.aembed(
        config=embeddings_config,
        input=EmbeddingsInput(text="Hello async world!")
    )

    print(f"Async embedding dimensions: {len(response.final_result.data[0].embedding)}")
    await async_service.aclose_http_connection()

await embed_async()